# Audio-engine — entraînement de tous les modèles en K-Fold (3 folds) + hard vote

**Avant de lancer :**
1. *Add Input* → ton dataset `safanasri002/dataset` (dossiers `physionet/` et `pascal/`)
2. Panneau de droite → *Session options* → **Internet : ON** (compte Kaggle vérifié par téléphone obligatoire)
3. *Run All*

```
features.npz ─┬─ 80 % TRAIN ──► K-Fold (3 folds) ──► choix des hyperparamètres (meilleur F2 moyen)
              │                                       └─► on GARDE les 3 modèles du meilleur candidat
              └─ 20 % TEST  ──► modèle 1 ─┐
                                modèle 2 ─┼─► HARD VOTE (2 sur 3) ──► note finale
                                modèle 3 ─┘
```
**Critère de choix : F2-score** -> le recall compte 4x plus que la précision (ne pas rater de malades),
mais un modèle qui dit toujours « anormal » est pénalisé.

À la fin, télécharge `resultats_kfold.zip` (onglet *Output*) : modèles + `mlflow.db` + courbes (`mlartifacts/`).

## 0. Installation et dossiers

In [ ]:
import socket

try:
    socket.create_connection(("pypi.org", 443), timeout=5)
    print("Internet OK")
except OSError:
    raise RuntimeError("Pas d'Internet : panneau de droite -> Session options -> Internet ON "
                       "(il faut avoir vérifié son numéro de téléphone sur Kaggle), puis relancer.")

In [ ]:
# Même version de MLflow que sur le PC -> mlflow.db s'ouvrira avec ton `mlflow ui` local
!pip install -q mlflow==3.18.0 skl2onnx onnxruntime

In [ ]:
import os

DATASET_PATH = "/kaggle/input/datasets/safanasri002/dataset/dataset"   # à adapter si ton chemin est différent
if not os.path.exists(DATASET_PATH):                                      # chemin Kaggle classique
    DATASET_PATH = "/kaggle/input/dataset/dataset"
print("Dataset :", DATASET_PATH, os.listdir(DATASET_PATH))

os.chdir("/kaggle/working")
os.makedirs("data", exist_ok=True)
# prepare_data.py lit data/raw/pascal et data/raw/physionet -> on pointe data/raw vers le dataset (sans copier)
if not os.path.exists("data/raw"):
    os.symlink(DATASET_PATH, "data/raw")
print(os.listdir("data/raw"))

### Serveur MLflow local

Les scripts enregistrent leurs runs via ce serveur : les images (courbes) vont dans `mlartifacts/`
avec des chemins **indépendants de la machine** -> elles s'afficheront aussi sur ton PC après téléchargement.

In [ ]:
import subprocess, time, urllib.request

mlflow_server = subprocess.Popen(
    ["mlflow", "server", "--backend-store-uri", "sqlite:///mlflow.db",
     "--artifacts-destination", "mlartifacts", "--host", "127.0.0.1", "--port", "5000"],
    stdout=open("mlflow_server.log", "w"), stderr=subprocess.STDOUT)

for _ in range(90):                      # attendre que le serveur réponde (max ~3 min)
    try:
        urllib.request.urlopen("http://127.0.0.1:5000/health")
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError(open("mlflow_server.log").read()[-2000:])

os.environ["MLFLOW_TRACKING_URI"] = "http://127.0.0.1:5000"   # lu par kfold.py
print("MLflow prêt")

## 1. Préparation des données : .wav → 40 MFCC → `data/features.npz`

In [ ]:
%%writefile prepare_data.py
"""ÉTAPE 1 — Data mining : à lancer UNE SEULE FOIS.

Lit tous les .wav, les nettoie, extrait les MFCC et sauvegarde le résultat
dans data/features.npz. Ensuite, les modèles chargent directement ce fichier.

    python prepare_data.py
"""
import csv
from pathlib import Path

import librosa
import numpy as np

RAW = Path("data/raw")              # où tu mets PASCAL et PhysioNet
OUTPUT = Path("data/features.npz")  # le résultat (petit fichier)


def preprocess(path):
    """Un fichier .wav -> un vecteur de 40 nombres (MFCC)."""
    y, sr = librosa.load(path, sr=2000)            # 1. rééchantillonnage à 2000 Hz
    y = librosa.util.fix_length(y, size=2000 * 5)  # 2. durée fixe : 5 secondes
    y = librosa.util.normalize(y)                  # 3. volume normalisé
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20, n_fft=256)
    return np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)])  # 4. moyenne + écart-type


def list_files():
    """Renvoie (chemin, label) avec label 0 = normal, 1 = pathologique."""
    files = []
    # PASCAL : le label est le nom du dossier (normal / extra / murmur)
    for folder, label in [("normal", 0), ("extra", 1), ("murmur", 1)]:
        for wav in (RAW / "pascal" / folder).glob("*.wav"):
            files.append((wav, label))

    # PhysioNet : le label est dans REFERENCE.csv  (-1 = normal, 1 = anormal)
    for ref in (RAW / "physionet").rglob("REFERENCE.csv"):
        for name, label in csv.reader(open(ref)):
            files.append((ref.parent / f"{name}.wav", 1 if label == "1" else 0))
    return files


if __name__ == "__main__":
    if OUTPUT.exists():
        print("Déjà fait :", OUTPUT, "(supprime-le pour recommencer)")
        exit()

    X, y = [], []
    for path, label in list_files():
        try:
            X.append(preprocess(path))
            y.append(label)
        except Exception:
            print("Fichier ignoré (corrompu) :", path)

    np.savez(OUTPUT, X=np.array(X), y=np.array(y))
    print(f"{len(y)} sons sauvegardés -> normal: {y.count(0)}, pathologique: {y.count(1)}")


In [ ]:
!python prepare_data.py

## 2. Outils K-Fold communs à tous les modèles (+ courbes d'apprentissage)

In [ ]:
%%writefile kfold.py
"""Outils communs à TOUS les modèles : données + validation croisée K-Fold + suivi MLflow.

Utilisé par train_svm.py et train_tree.py -> tous les modèles sont évalués sur les MÊMES folds.

Principe (doc scikit-learn "Cross-validation" + vote majoritaire) :
    features.npz ─┬─ 80 % TRAIN ──► K-Fold (3 folds) ──► choix des hyperparamètres
                  │                                       └─► on GARDE les 3 modèles du meilleur candidat
                  └─ 20 % TEST  ──► modèle 1 ─┐
                                    modèle 2 ─┼─► VOTE (2 sur 3) ──► note finale (une seule fois)
                                    modèle 3 ─┘

Courbes d'apprentissage (artefacts MLflow, dossier "courbes/") :
    - chaque run (candidat)  : learning_curves_folds.png -> une courbe train/validation par fold
    - le meilleur run        : learning_curve_final.png  -> moyenne ± écart-type des folds + score TEST du vote
"""
import os

import matplotlib

matplotlib.use("Agg")                  # pas de fenêtre : on génère juste des images pour MLflow
import matplotlib.pyplot as plt
import mlflow
import numpy as np
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score, fbeta_score, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split

N_FOLDS = 3                            # 3 modèles -> vote sans égalité possible (3-0 ou 2-1)
EXPERIMENT = "cardio-audio-kfold"
TRAIN_SIZES = [0.1, 0.25, 0.5, 0.75, 1.0]   # parts du train utilisées pour tracer les courbes d'apprentissage
CURVE_METRICS = ["f2", "recall_abnormal", "accuracy"]

# Couleurs des courbes : train = bleu, validation = orange, test = vert
COLOR_TRAIN, COLOR_VAL, COLOR_TEST, COLOR_INK = "#2a78d6", "#eb6834", "#1baf7a", "#52514e"


def load_split():
    """features.npz -> 80 % train (pour le K-Fold) / 20 % test (mis de côté pour la note finale)."""
    data = np.load("data/features.npz")
    X, y = data["X"], data["y"]
    return train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


def vote(models, X):
    """Vote majoritaire : chaque modèle prédit 0 (normal) ou 1 (anormal), la majorité gagne.

    Avec 3 modèles : au moins 2 votes « anormal » -> anormal, sinon normal.
    """
    preds = np.array([m.predict(X) for m in models])      # forme (nb_modèles, nb_sons)
    return (preds.sum(axis=0) > len(models) / 2).astype(int)


def compute_metrics(y_true, y_pred):
    return {"f2": fbeta_score(y_true, y_pred, beta=2),         # CRITÈRE DE CHOIX : recall compte 4x plus que la précision
            "recall_abnormal": recall_score(y_true, y_pred),   # % de pathologiques détectés
            "accuracy": accuracy_score(y_true, y_pred),
            "f1": f1_score(y_true, y_pred)}


# ─────────────────────────────── Courbes d'apprentissage ───────────────────────────────

def learning_curve_points(model, X_tr, y_tr, X_eval, y_eval):
    """Entraîne sur 10 %, 25 %, ... 100 % de X_tr et mesure le score sur ces données d'entraînement
    ET sur des données jamais vues (X_eval).

    Lecture :
    - les deux courbes montent et se rapprochent -> le modèle apprend bien
    - grand écart train >> eval                  -> surapprentissage
    - les deux restent basses                    -> sous-apprentissage
    """
    curve = {"n_samples": []}
    for split in ("train", "eval"):
        for name in CURVE_METRICS:
            curve[f"{split}_{name}"] = []

    for frac in TRAIN_SIZES:
        if frac < 1.0:   # sous-échantillon stratifié -> garde ~23 % d'anormaux même à 10 %
            X_sub, _, y_sub, _ = train_test_split(X_tr, y_tr, train_size=frac, stratify=y_tr, random_state=42)
        else:
            X_sub, y_sub = X_tr, y_tr
        m = clone(model).fit(X_sub, y_sub)
        curve["n_samples"].append(len(y_sub))
        for split, (Xs, ys) in {"train": (X_sub, y_sub), "eval": (X_eval, y_eval)}.items():
            metrics = compute_metrics(ys, m.predict(Xs))
            for name in CURVE_METRICS:
                curve[f"{split}_{name}"].append(float(metrics[name]))
    return curve


def _style(ax, title, ylabel=None):
    ax.set_title(title, fontsize=10, color=COLOR_INK)
    ax.set_xlabel("nb de sons d'entraînement", fontsize=8, color=COLOR_INK)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=9, color=COLOR_INK)
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.25)
    ax.tick_params(labelsize=8, colors=COLOR_INK)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)


def plot_fold_curves(fold_curves, title):
    """Une colonne par fold, une ligne par métrique : train (bleu) vs validation du fold (orange)."""
    n_rows, n_cols = len(CURVE_METRICS), len(fold_curves)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.2 * n_cols, 3 * n_rows), sharey=True, squeeze=False)
    for col, curve in enumerate(fold_curves):
        for row, name in enumerate(CURVE_METRICS):
            ax = axes[row][col]
            ax.plot(curve["n_samples"], curve[f"train_{name}"], "o-", color=COLOR_TRAIN, lw=2, ms=5, label="train")
            ax.plot(curve["n_samples"], curve[f"eval_{name}"], "o-", color=COLOR_VAL, lw=2, ms=5, label="validation")
            _style(ax, f"Fold {col + 1}" if row == 0 else "", name if col == 0 else None)
    axes[0][0].legend(fontsize=8, frameon=False, loc="lower right")
    fig.suptitle(title, fontsize=12, color=COLOR_INK)
    fig.tight_layout()
    return fig


def plot_final_curve(fold_curves, test_scores, title):
    """Meilleurs hyperparamètres : moyenne ± écart-type des folds + score du VOTE des modèles sur le TEST."""
    fig, axes = plt.subplots(1, len(CURVE_METRICS), figsize=(5.5 * len(CURVE_METRICS), 4), sharey=True)
    n_cv = np.array(fold_curves[0]["n_samples"])
    for ax, name in zip(axes, CURVE_METRICS):
        for split, color, label in (("train", COLOR_TRAIN, "train (moyenne K-Fold)"),
                                    ("eval", COLOR_VAL, "validation (moyenne K-Fold)")):
            values = np.array([c[f"{split}_{name}"] for c in fold_curves])
            mean, std = values.mean(axis=0), values.std(axis=0)
            ax.plot(n_cv, mean, "o-", color=color, lw=2, ms=5, label=label)
            ax.fill_between(n_cv, mean - std, mean + std, color=color, alpha=0.15, lw=0)   # bande ± écart-type
        ax.axhline(test_scores[name], ls="--", color=COLOR_TEST, lw=2,
                   label=f"TEST (vote des {len(fold_curves)} modèles) = {test_scores[name]:.2f}")
        _style(ax, name, "score")
    axes[0].legend(fontsize=8, frameon=False, loc="lower right")
    fig.suptitle(title, fontsize=12, color=COLOR_INK)
    fig.tight_layout()
    return fig


# ─────────────────────────────── K-Fold + MLflow ───────────────────────────────

def kfold_scores(model, X, y):
    """Entraîne et évalue le modèle sur N_FOLDS découpages -> renvoie les scores, la courbe ET le modèle de chaque fold."""
    # Stratifié = chaque fold garde la même proportion normal / anormal (~23 % d'anormaux)
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

    scores = {"f2": [], "recall_abnormal": [], "accuracy": [], "f1": [], "train_accuracy": []}
    curves, fold_models = [], []
    for train_idx, val_idx in skf.split(X, y):
        m = clone(model)                       # un modèle neuf à chaque fold (sinon il garde l'ancien entraînement)
        m.fit(X[train_idx], y[train_idx])
        fold_models.append(m)                  # on GARDE le modèle : il votera en inférence
        for name, value in compute_metrics(y[val_idx], m.predict(X[val_idx])).items():
            scores[name].append(value)
        scores["train_accuracy"].append(m.score(X[train_idx], y[train_idx]))   # pour voir le surapprentissage
        curves.append(learning_curve_points(model, X[train_idx], y[train_idx], X[val_idx], y[val_idx]))

    return {name: np.array(values) for name, values in scores.items()}, curves, fold_models


def train_with_kfold(family, candidates):
    """Teste chaque candidat en K-Fold, garde les N_FOLDS modèles du meilleur, note leur VOTE sur le TEST.

    family     : nom du modèle ("SVM", "DecisionTree", ...)
    candidates : liste de (nom_du_run, hyperparamètres, modèle_non_entraîné)
    Renvoie la liste des N_FOLDS modèles entraînés (un par fold) -> à faire voter avec vote().
    """
    X_train, X_test, y_train, y_test = load_split()
    # Par défaut : base locale mlflow.db. Sur Kaggle : le serveur MLflow lancé par le notebook (variable d'environnement)
    mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "sqlite:///mlflow.db"))
    mlflow.set_experiment(EXPERIMENT)

    # 1. K-Fold sur chaque candidat -> un run MLflow par candidat
    best = None
    for run_name, params, model in candidates:
        with mlflow.start_run(run_name=run_name) as run:
            scores, curves, fold_models = kfold_scores(model, X_train, y_train)

            mlflow.log_params({"model": family, "n_folds": N_FOLDS, **params})
            for name, values in scores.items():
                mlflow.log_metric(f"cv_{name}_mean", values.mean())
                mlflow.log_metric(f"cv_{name}_std", values.std())
                for fold, value in enumerate(values, start=1):    # courbe fold par fold dans MLflow
                    mlflow.log_metric(f"cv_{name}", value, step=fold)

            # Courbes d'apprentissage de chaque fold -> onglet "Artifacts" du run
            fig = plot_fold_curves(curves, f"{run_name} : courbes d'apprentissage par fold")
            mlflow.log_figure(fig, "courbes/learning_curves_folds.png")
            plt.close(fig)
            mlflow.log_dict({"train_sizes": TRAIN_SIZES, "folds": curves}, "courbes/learning_curves_folds.json")

            f2 = scores["f2"]
            print(f"{run_name:<22} F2 = {f2.mean():.3f} ± {f2.std():.3f}   "
                  f"recall = {scores['recall_abnormal'].mean():.3f}   acc = {scores['accuracy'].mean():.3f}   "
                  f"train acc = {scores['train_accuracy'].mean():.3f}")

            # 2. Garder le meilleur F2 MOYEN : priorité au recall (ne pas rater de malades)
            #    sans ignorer les fausses alarmes (un modèle qui dit toujours « anormal » a un F2 bas)
            if best is None or f2.mean() > best["f2"]:
                best = {"f2": f2.mean(), "name": run_name, "run_id": run.info.run_id,
                        "curves": curves, "fold_models": fold_models}

    # 3. Pas de ré-entraînement : les N_FOLDS modèles du meilleur candidat VOTENT, notés UNE fois sur le TEST
    models = best["fold_models"]
    test = compute_metrics(y_test, vote(models, X_test))
    # Pour comparer : score de chaque modèle seul sur le TEST (le vote doit être au moins aussi bon)
    alone = [compute_metrics(y_test, m.predict(X_test)) for m in models]

    with mlflow.start_run(run_id=best["run_id"]):
        mlflow.log_metrics({f"test_vote_{name}": value for name, value in test.items()})
        for i, metrics in enumerate(alone, start=1):
            mlflow.log_metrics({f"test_model{i}_{name}": value for name, value in metrics.items()})
        mlflow.set_tag("best", "true")
        mlflow.set_tag("inference", f"vote majoritaire de {len(models)} modèles")
        fig = plot_final_curve(best["curves"], test, f"Meilleur {family} : {best['name']}")
        mlflow.log_figure(fig, "courbes/learning_curve_final.png")
        plt.close(fig)

    print(f"\nMeilleur {family} : {best['name']} (F2 K-Fold = {best['f2']:.3f})")
    for i, metrics in enumerate(alone, start=1):
        print(f"  modèle {i} seul sur le TEST : " + "   ".join(f"{k} = {v:.3f}" for k, v in metrics.items()))
    print(f"Note finale du VOTE ({len(models)} modèles) sur le TEST : "
          + "   ".join(f"{k} = {v:.3f}" for k, v in test.items()))
    return models

## 3. SVM en K-Fold

In [ ]:
%%writefile train_svm.py
"""Entraîner le SVM en K-Fold (3 folds) avec plusieurs hyperparamètres -> chaque essai est sauvegardé dans MLflow.

    python train_svm.py
    mlflow ui --backend-store-uri sqlite:///mlflow.db     # puis ouvrir http://localhost:5000
"""
import joblib
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from kfold import train_with_kfold

# Les hyperparamètres à tester
C_values = [0.1, 1, 10]          # C petit = modèle simple, C grand = colle plus aux données
gamma_values = ["scale", 0.01]   # largeur du noyau RBF

# Le scaler est DANS le pipeline -> il est ré-appris sur chaque fold (pas de fuite de données)
candidates = [
    (f"SVM_C{C}_g{gamma}", {"C": C, "gamma": gamma},
     make_pipeline(StandardScaler(),
                   SVC(kernel="rbf", C=C, gamma=gamma, class_weight="balanced", probability=True)))
    for C in C_values for gamma in gamma_values
]

# On récupère les 3 modèles du meilleur candidat (un par fold) -> ils voteront en inférence
fold_models = train_with_kfold("SVM", candidates)
joblib.dump(fold_models, "data/svm_folds.joblib")
print(f"-> {len(fold_models)} modèles sauvegardés dans data/svm_folds.joblib (puis : python export_onnx.py)")

In [ ]:
!MLFLOW_DISABLE_AGENT_HINT=1 python train_svm.py

## 4. Arbre de décision en K-Fold

In [ ]:
%%writefile train_tree.py
"""Entraîner un arbre de décision en K-Fold (3 folds) avec plusieurs hyperparamètres -> MLflow.

Mêmes folds et même expérience que train_svm.py -> les deux modèles se comparent directement.

    python train_tree.py
    mlflow ui --backend-store-uri sqlite:///mlflow.db     # puis ouvrir http://localhost:5000
"""
import joblib
from sklearn.tree import DecisionTreeClassifier

from kfold import train_with_kfold

# Les hyperparamètres à tester
#    Pas de StandardScaler : un arbre compare chaque feature à un seuil, l'échelle ne change rien.
depth_values = [3, 5, 10, None]   # profondeur max : petit = règles simples, None = l'arbre pousse jusqu'au bout
leaf_values = [1, 10]             # nb min de sons par feuille : grand = évite d'apprendre par cœur

candidates = [
    (f"Tree_d{depth}_leaf{leaf}", {"max_depth": depth, "min_samples_leaf": leaf},
     DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf, class_weight="balanced", random_state=42))
    for depth in depth_values for leaf in leaf_values
]

# On récupère les 3 modèles du meilleur candidat (un par fold) -> ils voteront en inférence
fold_models = train_with_kfold("DecisionTree", candidates)
joblib.dump(fold_models, "data/tree_folds.joblib")
print(f"-> {len(fold_models)} modèles sauvegardés dans data/tree_folds.joblib")

In [ ]:
!MLFLOW_DISABLE_AGENT_HINT=1 python train_tree.py

## 5. Export ONNX des 3 modèles SVM (ils votent dans le serveur)

In [ ]:
%%writefile export_onnx.py
"""ÉTAPE 3 — Convertir les 3 modèles du K-Fold (svm_folds.joblib) au format ONNX.

ONNX = format universel : le serveur n'a plus besoin de scikit-learn, seulement d'onnxruntime (léger).
Un fichier ONNX par modèle -> le serveur les fait voter (2 sur 3).

    python export_onnx.py
"""
import joblib
import numpy as np
import onnxruntime as ort
from skl2onnx import to_onnx

from kfold import vote

# 1. Charger les modèles entraînés (un par fold)
models = joblib.load("data/svm_folds.joblib")
X = np.load("data/features.npz")["X"].astype(np.float32)

sessions = []
for i, model in enumerate(models, start=1):
    # 2. Convertir en ONNX (zipmap=False -> les probabilités sortent sous forme de tableau simple)
    path = f"data/cardio_model_{i}.onnx"
    onnx_model = to_onnx(model, X[:1], options={"zipmap": False})
    with open(path, "wb") as f:
        f.write(onnx_model.SerializeToString())

    # 3. Vérifier : chaque ONNX doit donner les mêmes prédictions que scikit-learn
    session = ort.InferenceSession(path)
    labels, _ = session.run(None, {session.get_inputs()[0].name: X[:100]})
    same = (labels == model.predict(X[:100])).mean()
    print(f"{path} créé | prédictions identiques à scikit-learn : {same:.0%}")
    sessions.append(session)

# 4. Vérifier le VOTE : ONNX (comme dans server.py) vs scikit-learn (comme dans kfold.py)
onnx_votes = np.array([s.run(None, {s.get_inputs()[0].name: X[:100]})[0] for s in sessions])
onnx_vote = (onnx_votes.sum(axis=0) > len(sessions) / 2).astype(int)
same = (onnx_vote == vote(models, X[:100])).mean()
print(f"Vote des {len(sessions)} modèles ONNX identique au vote scikit-learn : {same:.0%}")

In [ ]:
!python export_onnx.py

## 6. Tableau récapitulatif (lu depuis MLflow)

In [ ]:
import mlflow

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
runs = mlflow.search_runs(experiment_names=["cardio-audio-kfold"])
cols = ["tags.mlflow.runName", "params.model", "params.n_folds",
        "metrics.cv_f2_mean", "metrics.cv_f2_std", "metrics.cv_recall_abnormal_mean",
        "metrics.cv_accuracy_mean", "metrics.cv_train_accuracy_mean",
        "metrics.test_vote_f2", "metrics.test_vote_recall_abnormal", "metrics.test_vote_accuracy"]
runs = runs[runs["params.n_folds"] == "3"]             # seulement les runs de cette version (3 folds + F2)
runs[[c for c in cols if c in runs]].sort_values("metrics.cv_f2_mean", ascending=False).round(3)

## 7. Tout regrouper pour le téléchargement

In [ ]:
mlflow_server.terminate()   # fermer proprement mlflow.db avant de le copier
mlflow_server.wait()
!zip -qr resultats_kfold.zip data/features.npz data/svm_folds.joblib data/tree_folds.joblib data/cardio_model_*.onnx mlflow.db mlartifacts
!ls -lh resultats_kfold.zip